# RNN Quest

In [42]:
import lpnlp

lab = lpnlp.start(
    email="sviatoslav.humeniuk.shi.2023@lpnu.ua",
    lab="quest_rnn"
)

Ваше завдання: https://static.lp-nlp.com/quest-rnn/df489bc07837.zip. Удачі! █


1. Завантажте пакет із завданням за посиланням вище ^
2. Розпакуйте архів
3. Прочитайте README та виконайте завдання
  - Завдання не залежить від мови програмування
  - Завдання не залежить від фреймоворку. Pytorch, NumPy, Keras або взагалі без сторонніх бібліотек - все можливо
4. Відправте відповідь нижче

#### Зчитування ваг

In [43]:
import json
from pathlib import Path

import numpy as np

WEIGHT_NAMES = [
    "embedding.weight",
    "W_h.weight", "W_h.bias",
    "U_h.weight", "U_h.bias",
    "W_y.weight", "W_y.bias",
]


def read_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


def load_weights(weights_dir: Path) -> tuple[list[str], dict[str, np.ndarray]]:
    vocab = read_json(weights_dir / "vocab.json")
    weights = {name: np.array(read_json(weights_dir / f"{name}.json")) for name in WEIGHT_NAMES}
    return vocab, weights

In [44]:
vocabulary, weights = load_weights(Path("source"))
character2index = {ch: i for i, ch in enumerate(vocabulary)}

#### Цикл жадібного декодування 

In [45]:
MAX_ITERATIONS = 200


def greedy_decode(vocab: list[str], weights: dict[str, np.ndarray], max_length: int = MAX_ITERATIONS) -> str:
    char2index = {ch: i for i, ch in enumerate(vocab)}
    h = np.zeros(weights["U_h.weight"].shape[0])
    current_token = '['
    generated_characters = []

    for _ in range(max_length):
        x = weights["embedding.weight"][char2index[current_token]]

        h = np.tanh(
            weights["W_h.weight"] @ x + weights["W_h.bias"]
            + weights["U_h.weight"] @ h + weights["U_h.bias"]
        )
        y = weights["W_y.weight"] @ h + weights["W_y.bias"]

        next_character = vocab[int(np.argmax(y))]
        if next_character == ']':
            break

        generated_characters.append(next_character)
        current_token = next_character

    return "".join(generated_characters)


print(greedy_decode(vocabulary, weights))

Oct 12, city name: https://www.youtube.com/watch?v=cRMj7khQKKM


In [46]:
lab.answer("Barcelona")

Відповідь правильна ✅
You did it! 🚀 Next step: https://tally.so/r/wMXM0p


#### Завдання з зірочкою: Тренування власної RNN

In [47]:
import torch
import torch.nn as nn

torch.manual_seed(66)

SECRET_MESSAGE = "Alexander the Great"
FULL_TEXT = f"[{SECRET_MESSAGE}]"

for character in FULL_TEXT:
    assert character in character2index, f"Символ '{character}' відсутній у vocab.json."

token_indices = [character2index[character] for character in FULL_TEXT]
inputs = torch.tensor(token_indices[:-1], dtype=torch.long)
targets = torch.tensor(token_indices[1:], dtype=torch.long)

In [48]:
class ExtraTaskRNN(nn.Module):
    def __init__(self, vocabulary_size=132, embedding_dimension=32, hidden_dimension=160):
        super().__init__()
        self.hidden_dimension = hidden_dimension
        self.embedding = nn.Embedding(vocabulary_size, embedding_dimension)
        self.W_h = nn.Linear(embedding_dimension, hidden_dimension, bias=True)
        self.U_h = nn.Linear(hidden_dimension, hidden_dimension, bias=True)
        self.W_y = nn.Linear(hidden_dimension, vocabulary_size, bias=True)

    def forward_step(self, token_index, h_previous):
        x = self.embedding(token_index)
        h_t = torch.tanh(self.W_h(x) + self.U_h(h_previous))
        y_t = self.W_y(h_t)
        return y_t, h_t

model = ExtraTaskRNN(vocabulary_size=len(vocabulary))

In [49]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=0.0)
criterion = nn.CrossEntropyLoss()

model.train()

for epoch in range(1, MAX_ITERATIONS + 1):
    optimizer.zero_grad()
    h = torch.zeros(1, model.hidden_dimension)
    loss = torch.tensor(0.0)

    for t in range(len(inputs)):
        y_t, h = model.forward_step(inputs[t].unsqueeze(0), h)
        loss += criterion(y_t, targets[t].unsqueeze(0))

    loss.backward()
    optimizer.step()

In [50]:
import zipfile

output_dir = Path("extra")
output_dir.mkdir(exist_ok=True)

def save_json(filename, tensor_data):
    with open(output_dir / filename, "w", encoding="utf-8") as f:
        json.dump(tensor_data.detach().cpu().numpy().tolist(), f)

save_json("embedding.weight.json", model.embedding.weight)
save_json("W_h.weight.json", model.W_h.weight)
save_json("W_h.bias.json", model.W_h.bias)
save_json("U_h.weight.json", model.U_h.weight)
save_json("U_h.bias.json", model.U_h.bias)
save_json("W_y.weight.json", model.W_y.weight)
save_json("W_y.bias.json", model.W_y.bias)

with open(output_dir / "vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocabulary, f, ensure_ascii=False, indent=2)

In [51]:
print(greedy_decode(*load_weights(output_dir)))

Alexander the Great


In [52]:
zip_path = Path("ШІ43_Гуменюк_Святослав_bonus_quest.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file in output_dir.glob("*.json"):
        zipf.write(file, arcname=file.name)

print(f"Архів створено: '{zip_path}'")

Архів створено: 'ШІ43_Гуменюк_Святослав_bonus_quest.zip'
